In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import os
from pyprojroot import here

# ==============================================================================
# 🎯 CONFIGURATION
# ==============================================================================
# --- Cell 2: โหลดข้อมูลและกำหนด Target ---
# โหลดข้อมูล Panel Data ของเกษตรกร
base_path = here()
FILE_2007 = os.path.join(base_path, "datas", "wave-1-2007-shocksclean.dta")
# df = pd.read_stata(file_path,convert_categoricals=False)

# print(f"Loaded Wave 1 (2007) Baseline successfully: {len(df_07)} rows")

#FILE_2007 = 'wave-1-shocksclean.dta'
OUTPUT_FILE = 'shocks_2007_wide_panel.csv'

# ==============================================================================
# 🎯 
# ==============================================================================


# ตั้งค่าธีมกราฟ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


def load_and_profile_stata(file_path: str) -> pd.DataFrame:
    """โหลดไฟล์ Stata (.dta) บังคับรหัสตัวตน และตรวจสอบข้อมูลเบื้องต้น"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"ไม่พบไฟล์ดิบ: {file_path}")
        
    print(f"--- 🚀 เริ่มกระบวนการสกัดข้อมูลและแปลงเป็น Wide สำหรับ: {file_path} ---")
    
    # โหลดไฟล์ Stata (ปิดการแปลงกะทันหันเพื่อป้องกันคอลัมน์พัง)
    df = pd.read_stata(file_path, convert_categoricals=False)
    print(f"[Profile] ขนาดไฟล์ดิบปี 2007: {df.shape[0]} แถว | {df.shape[1]} คอลัมน์")
    
    return df


def determine_primary_key(df: pd.DataFrame) -> str:
    """ค้นหาคีย์หลักของครัวเรือน (Priority: qid -> hhid -> interview__key)"""
    for key in ['qid', 'hhid', 'interview__key']:
        if key in df.columns:
            print(f"-> ใช้คอลัมน์ '{key}' เป็นคีย์หลักในการจัดกลุ่มครัวเรือน")
            return key
    raise KeyError("ไม่พบคอลัมน์คีย์หลัก (qid, hhid, interview__key) ในไฟล์นี้")


def clean_wave1_baseline(df: pd.DataFrame, id_key: str) -> pd.DataFrame:
    """ล้างข้อมูลขั้นพื้นฐานโดยไม่เปลี่ยนชื่อคอลัมน์เดิม"""
    df_cleaned = df.copy()
    
    # 1. จัดการค่าว่างมาตรฐานของระบบ Stata
    df_cleaned = df_cleaned.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 2. จัดการมูลค่าความเสียหายในคอลัมน์เดิม (_x31005) ถ้ามี
    if '_x31005' in df_cleaned.columns:
        df_cleaned['_x31005'] = df_cleaned['_x31005'].astype(str).str.replace('none', '0', case=False).str.strip()
        df_cleaned['_x31005'] = pd.to_numeric(df_cleaned['_x31005'], errors='coerce').fillna(0)
        
    # 3. คลีนเฉพาะแถวที่ไม่มีรหัสตัวตนออก ป้องกันระบบ Pivot ล่ม
    df_cleaned = df_cleaned.dropna(subset=[id_key])
    
    # 4. แปลงรหัสตัวตนให้เป็น String ชัวร์ที่สุด
    df_cleaned[id_key] = df_cleaned[id_key].astype(str)
    
    return df_cleaned


def reshape_to_wide(df: pd.DataFrame, id_key: str) -> pd.DataFrame:
    """กางแถวซ้ำ (Multiple Shocks) ออกไปทางขวาดื้อๆ พร้อมห้อยท้ายด้วยลำดับเหตุการณ์"""
    # ระบุคอลัมน์ดั้งเดิมทั้งหมดที่จะกาง (ไม่แตะชื่อเดิมเลย)
    target_cols = [col for col in df.columns if col != id_key]
    
    # สร้างตัวนับลำดับภัยพิบัติ (Vectorized cumcount)
    df['shock_seq'] = df.groupby(id_key).cumcount() + 1
    max_shocks = df['shock_seq'].max()
    print(f"🌪️ ครัวเรือนในปี 2007 เคยเจอภัยพิบัติสูงสุดพร้อมกัน = {max_shocks} ครั้ง")
    
    # ใช้ Pivot กางออกไปทางขวาตรงๆ ดื้อๆ
    df_wide = df.pivot(
        index=id_key,
        columns='shock_seq',
        values=target_cols
    )
    
    # ต่อชื่อคอลัมน์ดั้งเดิมเข้ากับเลขลำดับ เช่น _x31005n_1, _x31005n_2
    # รักษาสภาพ ตัวเล็ก/ตัวใหญ่ ของคอลัมน์ดั้งเดิมไว้ครบถ้วน 100%
    df_wide.columns = [f"{orig_col}_{seq}" for orig_col, seq in df_wide.columns]
    
    return df_wide.reset_index()


def main():
    # 1. โหลดข้อมูลดิบ Stata
    df_raw = load_and_profile_stata(FILE_2007)
    
    # 2. ค้นหาคีย์หลัก
    id_key = determine_primary_key(df_raw)
    
    # 3. ล้างข้อมูลพื้นฐาน (คงชื่อเดิมไว้)
    df_cleaned = clean_wave1_baseline(df_raw, id_key)
    
    # 4. แปลงร่างเป็น Wide Format กางออกทางขวา
    df_wide = reshape_to_wide(df_cleaned, id_key)
    
    # 5. ตรวจสอบความถูกต้อง (Data Integrity Check)
    print(f"\n✨ กระบวนการแปลงข้อมูลเรียบร้อย!")
    print(f"📊 ขนาดข้อมูลใหม่ (Wide): {df_wide.shape[0]} แถว | {df_wide.shape[1]} คอลัมน์")
    
    if df_wide[id_key].duplicated().sum() == 0:
        print("✅ Verification: ยืนยันข้อมูลเหลือ 1 ครัวเรือนต่อ 1 แถวอย่างถูกต้อง")
    else:
        print("⚠️ Warning: พบแถวซ้ำของครัวเรือนในผลลัพธ์!")

    # 6. บันทึกผลลัพธ์
    df_wide.to_csv(OUTPUT_FILE, index=False)
    print(f"💾 บันทึกไฟล์สำเร็จในชื่อ: {OUTPUT_FILE}\n")
    
    # 7. พลอตสรุปจำนวนนับของ Shock สูงสุดที่เจอในปีนี้ (ดูว่ากระจายตัวยังไง)
    sns.countplot(data=df_cleaned, x='shock_seq', palette='magma')
    plt.title('Distribution of Shock Sequences per Household (Wave 1 - 2007)')
    plt.xlabel('Shock Sequence (1st, 2nd, 3rd, ...)')
    plt.ylabel('Household Count')
    plt.show()


if __name__ == '__main__':
    main()